# 12.c — Ad hoc test: robustness of the crop model on ERA5 (spatial blocks, halves of the period)

**Ad hoc test, outside the suitability chain**: it reads the outputs of 0.1, 0.5, 1.1.d, 2.g and 3.g (and of 12.a for a
comparison) and the daily ERA5, and writes only in `<out_root>/test_era5_robustness/<route>/<set>/`.

**Question.** The model was judged good on ERA5 through the out-of-sample scores of the random stratified folds of
3.g / 4.g (8.2: AUC 0.964–0.986, TSS at 0.5 0.82–0.91). Two things are not covered:
1. **Spatial autocorrelation.** Neighbouring cells have nearly the same climate and often the same label; a random fold
   keeps, for every held-out cell, its neighbours in the training. The scores then measure an interpolation and are
   optimistic for the cells far from the training data (Roberts et al. 2017, *Ecography* 40, 913–929; Ploton et al.
   2020, *Nat. Commun.* 11, 4540). Remedy: cross-validation by **spatial blocks** (Valavi et al. 2019, *Methods Ecol.
   Evol.* 10, 225–232), the held-out cells grouped in blocks larger than the range of the autocorrelation, and, as the
   harshest case, **whole continents** (transferability; Wenger & Olden 2012, *Methods Ecol. Evol.* 3, 260–267).
2. **The ERA5 period.** The suitability "today" rests on the means of 33 seasons (harvest years 1982–2014). How much
   does it change with **the other half of the period** (1982–1997 against 1998–2014)?

**Method.**
1. *Spatial blocks* (section 1). The training table of 3.g, the final cost and gamma, the class weights of 3.g (absence
   weight = presences / absences of the training fold), Platt probability brought to equal priors with the class counts
   of the training fold (`balanced_proba`). Schemes: **random** = the five stratified folds of 3.g (seed 42, the fold
   models of 4.g); **blocks of 5°, 10°, 20°** (latitude × longitude) assigned at random to five folds of similar size;
   **continents** = seven groups of AR6 land regions, one held out at a time. For each held-out cell: the probability,
   and whether it is novel to the training fold (MESS < 0, NT2 > 1 or kernel support < 0.01, as 4.g). Scores of the
   pooled held-out cells: AUC, TSS (sensitivity + specificity − 1) at p ≥ 0.5 for the SVM alone and for the **climate
   rule** (p ≥ 0.5, not novel, ECOCROP TMIN of 4.g), the threshold of maximum TSS, the share of the MIRCA2000 area of
   the crop kept by the rule. Also per continent.
2. *Halves of ERA5* (section 2). BIO variables of the harvest years 1982–1997 (H1, 16 seasons), 1998–2014 (H2, 17) and
   1982–2014 (full, check: = 1.1.d) with the code of 12.a (= 1.1.d, no_rad); final suitability of 4.g (function of
   12.a: cross-fitting, novelty, TMIN, soil ≥ 10, low and high input). Share of today's crop area suitable (points of
   the MIRCA2000 area of the crop), suitable land per region, cells that change class between H1 and H2, and the TSS
   of the training cells with the climate of each half.

**Reading.**
- *Blocks*: a small loss from random to 5–10° blocks = the skill is not an artefact of the autocorrelation. A large
  loss for whole continents with most of the held-out cells novel = the model is not meant to extrapolate, which is
  why novel climates are set to not suitable (4.g); the scores on the non-novel cells say whether the rule protects it.
- *Halves*: H2 − H1 mixes the sampling of 16–17 seasons and the observed warming between the two halves (about
  0.4 °C), so it is an upper bound of the sampling noise. The standard error of the 33-season baseline is about
  |H2 − H1| / 2 (two independent halves: var(H2 − H1) ≈ 4 var(full)), to compare with the effects of the AMOC
  weakening (12.a).

Limits: one predictor set (A_60d, no_rad); continents differ in their crops and climates, so their scores mix the
transferability and the composition of the held-out set; the temporal transferability with other MIRCA-OS years is not
tested here.

In [ ]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import time
import numpy as np, pandas as pd, xarray as xr, joblib
import matplotlib.pyplot as plt
from pathlib import Path
from joblib import Parallel, delayed
from scipy.spatial import cKDTree
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
import warnings; warnings.filterwarnings('ignore'); os.environ["PYTHONWARNINGS"] = "ignore"   # also in the joblib workers

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu
CFG = cu.load_config()

# ---- parameters: the final set of 1.1.d, 3.g and 4.g
CHOSEN_VARIANT = "A_60d"
CHOSEN_RAD = "no_rad"
CELL_AREA_THR = 0.0005
CROPS = ["Maize_rf", "Potato_rf", "Soybean_rf", "Rice_rf", "Wheat_rf", "Wheat-winter_rf", "Wheat-spring_rf"]
ERA5_NAME, ERA5_PERIOD = "ERA5", ("1981", "2014")     # training climate (1.1.d)
CLIMATE = "era5"                                      # daily ERA5 on the EC-Earth3 grid (dataset of the configuration, 1.1.d)
LEVELS = ["low", "high"]          # GAEZ input level of the soil rating (2.g)
KERNEL_EPS = 0.01                 # novel where the kernel support < KERNEL_EPS (4.g)
N_JOBS = min(5, os.cpu_count() or 1)
N_FOLDS, SEED = 5, 42             # as 3.g
BLOCK_DEGS = [5, 10, 20]          # sizes of the spatial blocks (degrees of latitude and longitude)
CONTINENTS = {                    # groups of AR6 land regions (Iturbide et al. 2020), one held out at a time
    "North America": ["GIC", "NWN", "NEN", "WNA", "CNA", "ENA", "NCA"],
    "Latin America": ["SCA", "CAR", "NWS", "NSA", "NES", "SAM", "SWS", "SES", "SSA"],
    "Europe and Mediterranean": ["NEU", "WCE", "EEU", "MED"],
    "Africa": ["SAH", "WAF", "CAF", "NEAF", "SEAF", "WSAF", "ESAF", "MDG"],
    "North and Central Asia": ["RAR", "WSB", "ESB", "RFE", "WCA", "ECA", "TIB", "ARP"],
    "South and East Asia": ["EAS", "SAS", "SEA"],
    "Oceania": ["NAU", "CAU", "EAU", "SAU", "NZ"]}
OVERWRITE = False                 # False: held-out predictions and half BIO already written are read back
SECTIONS = os.environ.get("TEST12C", "blocks,halves").split(",")   # "blocks", "halves" or both
assert CHOSEN_RAD == "no_rad", "no_rad only"

SET = f"{CHOSEN_VARIANT}_{CHOSEN_RAD}"
ROUTE = cu.route(CELL_AREA_THR)
TAG = f"{ERA5_NAME}_{ERA5_PERIOD[0]}_{ERA5_PERIOD[1]}_ecearth3"
ERA5_DIR = cu.out_dir(CFG, "1_biovariables", f"{ERA5_NAME}_{ROUTE}", CHOSEN_VARIANT)
MODELS = cu.out_dir(CFG, "3_svm_train", f"{ERA5_NAME}_{ROUTE}", SET)
SOIL_DIR = cu.out_dir(CFG, "2_soil_suitability")
VAR12A = cu.out_dir(CFG, "test_internal_variability", ROUTE, SET)       # 12.a (read only, for the comparison)
OUT = cu.out_dir(CFG, "test_era5_robustness", ROUTE, SET)              # the only folder written
OOF_OUT = OUT / "heldout"; OOF_OUT.mkdir(exist_ok=True)
BIO_OUT = OUT / "bio"; BIO_OUT.mkdir(exist_ok=True)

GRID = cu.svm_grid(CFG)
LAND = cu.land_mask(CFG).values
I, J = np.nonzero(LAND)
LAT_RAD = np.radians(GRID["lat"][I])
LAND_MHA = np.nan_to_num(cu.land_area_ha(CFG)) * LAND / 1e6
DT = cu.tmin_warming(CFG)
LAT2, LON2 = np.meshgrid(GRID["lat"], GRID["lon"], indexing="ij")
LON180 = np.mod(LON2 + 180, 360) - 180
REGIONS = {"global": LAND,                                                   # as 6.b
           "Europe": LAND & (LAT2 >= 35) & (LAT2 <= 72) & (LON180 >= -25) & (LON180 <= 45),
           "north of 50 N": LAND & (LAT2 > 50),
           "23.5 - 50 N": LAND & (LAT2 > 23.5) & (LAT2 <= 50),
           "tropics": LAND & (np.abs(LAT2) <= 23.5),
           "south of 23.5 S": LAND & (LAT2 < -23.5)}
SOIL = {lv: xr.open_dataset(SOIL_DIR / f"soil_suitability_rf_{lv}_ecearth3.nc").load() for lv in LEVELS}
era5_file = lambda crop: ERA5_DIR / f"BIOvariables_{TAG}_{crop}.nc"
half_file = lambda part, crop: BIO_OUT / f"BIOvariables_ERA5_{part}_{crop}.nc"
with xr.open_dataset(era5_file(CROPS[0])) as d0:
    COLS = [v for v in d0.data_vars if v.startswith("BIO") and not v.endswith("e") and v not in ("BIO03", "BIO08")]

# continent of every land cell: AR6 land regions grouped; cells outside the AR6 polygons (coasts, islands) take the
# continent of the nearest land cell that has one
CONT_NAMES = list(CONTINENTS)
try:
    import regionmask
    AR6 = regionmask.defined_regions.ar6.land
    reg = AR6.mask(GRID["lon"], GRID["lat"]).values
    ab2c = {a: k for k, c in enumerate(CONT_NAMES) for a in CONTINENTS[c]}
    num2c = {n: ab2c.get(a, -1) for n, a in zip(AR6.numbers, AR6.abbrevs)}
    CONT = np.where(np.isfinite(reg), np.vectorize(lambda n: num2c.get(int(n), -1) if np.isfinite(n) else -1)(reg), -1)
except Exception as e:                       # no regionmask: boxes of latitude and longitude (rough)
    print("regionmask not available, continents from boxes:", e)
    lo, la = LON180, LAT2
    CONT = np.select([(lo < -30) & (la >= 13), lo < -30, (lo < 60) & (la >= 35), (lo < 60) & ~((lo > 35) & (la > 12)),
                      lo < 60, la >= 45, la > -10, True], [0, 1, 2, 3, 4, 4, 5, 6])
xyz = lambda la, lo: np.column_stack([np.cos(np.radians(la)) * np.cos(np.radians(lo)),
                                      np.cos(np.radians(la)) * np.sin(np.radians(lo)), np.sin(np.radians(la))])
known = LAND & (CONT >= 0)
todo = LAND & (CONT < 0)
if todo.any():
    _, ix = cKDTree(xyz(LAT2[known], LON180[known])).query(xyz(LAT2[todo], LON180[todo]))
    CONT[todo] = CONT[known][ix]
print(f"set {SET} | sections {SECTIONS} | blocks {BLOCK_DEGS} deg | written only in {OUT}")
print({c: int((LAND & (CONT == k)).sum()) for k, c in enumerate(CONT_NAMES)}, "land cells per continent")

## 1. Spatial-block cross-validation of the SVM

For each crop and scheme, the held-out probability (equal priors) and novelty of every training cell, written to
`heldout/heldout_<scheme>_<crop>.csv` (read back unless `OVERWRITE`). The random scheme uses the fold models of 4.g
(`cu.crossfit_models`, the folds of 3.g): its scores reproduce 8.2.

In [ ]:
def fit_fold(X, y, tr, cost, gamma):
    # as cu.crossfit_models: scaler and class weights of the training fold, Platt probability
    s = StandardScaler().fit(X[tr])
    cw = {1: 1.0, 0: (y[tr] == 1).sum() / (y[tr] == 0).sum()}
    return s, SVC(kernel="rbf", C=cost, gamma=gamma, class_weight=cw, probability=True, random_state=SEED).fit(s.transform(X[tr]), y[tr])


def heldout(X, y, tr, te, fitted=None, cost=None, gamma=None, prior=None):
    # held-out cells te: probability at equal priors (class counts of the training fold, or `prior`) and novelty
    # against the training fold (MESS < 0, NT2 > 1, kernel support < KERNEL_EPS, as 4.g)
    s, m = fitted if fitted is not None else fit_fold(X, y, tr, cost, gamma)
    Xs = s.transform(X[te])
    n1, n0 = prior if prior is not None else ((y[tr] == 1).sum(), (y[tr] == 0).sum())
    p = cu.balanced_proba(m.predict_proba(Xs)[:, 1], n1, n0)
    ref = X[tr]
    nov = (cu.mess(X[te], ref)[0] < 0) | (cu.nt2(X[te], ref) > 1) | (cu.kernel_support(m)(Xs) < KERNEL_EPS)
    return te, p, nov


def block_folds(lat, lon, deg, y):
    # blocks of deg x deg assigned to N_FOLDS folds: blocks in random order (SEED), each to the fold with the fewest cells
    b = np.floor((lat + 90) / deg).astype(int) * 1000 + np.floor((np.mod(lon + 180, 360)) / deg).astype(int)
    ub, inv, cnt = np.unique(b, return_inverse=True, return_counts=True)
    order = np.random.default_rng(SEED).permutation(ub.size)
    size, fold_of = np.zeros(N_FOLDS), np.empty(ub.size, int)
    for k in order:
        f = int(np.argmin(size)); fold_of[k] = f; size[f] += cnt[k]
    f = fold_of[inv]
    return [(np.nonzero(f != k)[0], np.nonzero(f == k)[0]) for k in range(N_FOLDS)], f


def scores(y, p, nov, tmin_ok, area, sel=None):
    # pooled held-out scores; sel = subset of cells
    sel = (np.ones(y.size, bool) if sel is None else sel) & np.isfinite(p)     # (a continent without training data: not predicted)
    y, p, nov, tmin_ok, area = y[sel], p[sel], nov[sel], tmin_ok[sel], area[sel]
    pres = y == 1
    out = {"cells": int(sel.sum()), "presences": int(pres.sum()), "novel (%)": 100 * nov.mean() if y.size else np.nan}
    if pres.all() or (~pres).all():
        return out
    svm = p >= cu.CLIMATE_P
    rule = svm & ~nov & tmin_ok
    for name, s in (("SVM", svm), ("rule", rule)):
        sens, spec = s[pres].mean(), (~s[~pres]).mean()
        out[f"TSS {name}"] = sens + spec - 1
        out[f"sens {name}"], out[f"spec {name}"] = sens, spec
    out["AUC"] = roc_auc_score(y, p)
    out["best threshold"], out["max TSS"] = cu.tss_threshold(p, y)[:2]
    out["MIRCA area kept by the rule (%)"] = 100 * area[pres & rule].sum() / max(area[pres].sum(), 1e-9)
    ok = ~nov
    if (y[ok] == 1).any() and (y[ok] == 0).any():
        s = rule[ok]
        out["TSS rule, non-novel cells"] = s[y[ok] == 1].mean() + (~s[y[ok] == 0]).mean() - 1
    return out


SCHEMES = ["random", *[f"blocks {d} deg" for d in BLOCK_DEGS], "continents"]
if "blocks" in SECTIONS:
    rows, rows_c, FOLDMAP = [], [], {}
    for crop in CROPS:
        t0 = time.time()
        bundle = joblib.load(MODELS / f"svm_model_{TAG}_{crop}.joblib")
        cols, cost, gamma = bundle["bio_cols"], bundle["best_cost"], bundle["best_gamma"]
        tr_tab = pd.read_csv(ERA5_DIR / f"training_dataset_{TAG}_{crop}.csv")
        X, y = tr_tab[cols].values.astype(float), tr_tab["label"].values.astype(int)
        ii = np.searchsorted(GRID["lat"], tr_tab.lat.values - 1e-6); jj = np.searchsorted(GRID["lon"], tr_tab.lon.values - 1e-6)
        cont = CONT[ii, jj]
        area = np.nan_to_num(cu.crop_mask(CFG, crop)["area_ha"].values)[ii, jj]
        with xr.open_dataset(era5_file(crop)) as ds:
            tg = np.asarray(cu.growing_temperature(ds, cu.ECOCROP_TMIN[crop][1]), float)
        tmin_ok = cu.climate_suitable(np.ones(LAND.shape), tg, crop, DT)[ii, jj]        # the ECOCROP TMIN of the rule
        for scheme in SCHEMES:
            f = OOF_OUT / f"heldout_{scheme.replace(' ', '_')}_{crop}.csv"
            if f.exists() and not OVERWRITE:
                H = pd.read_csv(f)
                assert len(H) == len(y) and (H.label.values == y).all(), f"{f.name}: training table changed, rerun with OVERWRITE"
            else:
                if scheme == "random":
                    Fd = cu.crossfit_models(MODELS / f"folds_{TAG}_{crop}.joblib", X, y, bundle, n_jobs=N_JOBS)
                    folds = [(np.setdiff1d(np.arange(y.size), te), te) for te in Fd["test_idx"]]
                    fold = np.empty(y.size, int)
                    for k, te in enumerate(Fd["test_idx"]):
                        fold[te] = k
                    res = [heldout(X, y, tr, te, fitted=fm, prior=(bundle["n_class1"], bundle["n_class0"]))   # as 4.g
                           for (tr, te), fm in zip(folds, Fd["fits"])]
                else:
                    if scheme == "continents":
                        fold = cont.copy()
                        folds = [(np.nonzero(fold != k)[0], np.nonzero(fold == k)[0]) for k in range(len(CONT_NAMES))]
                        folds = [(tr, te) for tr, te in folds if te.size and (y[tr] == 1).any() and (y[tr] == 0).any()]
                    else:
                        folds, fold = block_folds(tr_tab.lat.values, tr_tab.lon.values, float(scheme.split()[1]), y)
                    res = Parallel(n_jobs=N_JOBS)(delayed(heldout)(X, y, tr, te, cost=cost, gamma=gamma) for tr, te in folds)
                p, nov = np.full(y.size, np.nan), np.zeros(y.size, bool)
                for te, pt, nt in res:
                    p[te], nov[te] = pt, nt
                H = pd.DataFrame({"lat": tr_tab.lat, "lon": tr_tab.lon, "label": y, "fold": fold, "continent": cont,
                                  "p": p, "novel": nov})
                H.to_csv(f, index=False)
            p, nov = H.p.values, H.novel.values.astype(bool)
            FOLDMAP[(crop, scheme)] = H
            rows.append({"crop": crop, "scheme": scheme, **scores(y, p, nov, tmin_ok, area)})
            for k, c in enumerate(CONT_NAMES):
                rows_c.append({"crop": crop, "scheme": scheme, "continent": c, **scores(y, p, nov, tmin_ok, area, cont == k)})
            print(f"{crop} {scheme}: TSS rule {rows[-1].get('TSS rule', np.nan):.3f}, AUC {rows[-1].get('AUC', np.nan):.3f}, "
                  f"novel {rows[-1]['novel (%)']:.1f} % ({(time.time() - t0) / 60:.1f} min)", flush=True)
    BLK = pd.DataFrame(rows).set_index(["crop", "scheme"])
    BLKC = pd.DataFrame(rows_c).set_index(["crop", "scheme", "continent"])
    BLK.to_csv(OUT / "block_cv_scores.csv"); BLKC.to_csv(OUT / "block_cv_scores_by_continent.csv")
    with pd.option_context("display.max_rows", 400, "display.width", 250):
        print("Pooled held-out scores (rule = p >= 0.5, not novel, ECOCROP TMIN)")
        display(BLK[["cells", "novel (%)", "AUC", "TSS SVM", "TSS rule", "TSS rule, non-novel cells", "sens rule", "spec rule",
                     "best threshold", "max TSS", "MIRCA area kept by the rule (%)"]].round(3))
        print("Loss of TSS (rule) against the random folds")
        display((BLK["TSS rule"].unstack("scheme")[SCHEMES].sub(BLK["TSS rule"].unstack("scheme")["random"], axis=0)).round(3))
        print("Continents held out: scores per continent")
        display(BLKC.xs("continents", level="scheme")[["cells", "presences", "novel (%)", "AUC", "TSS rule",
                                                       "TSS rule, non-novel cells", "MIRCA area kept by the rule (%)"]].round(3))

## 1b. Figures: scores by scheme; continents and blocks

In [ ]:
if "blocks" in SECTIONS:
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
    for ax, met in zip(axes, ["AUC", "TSS rule", "MIRCA area kept by the rule (%)"]):
        T = BLK[met].unstack("scheme")[SCHEMES]
        for crop in T.index:
            ax.plot(range(len(SCHEMES)), T.loc[crop].values, "o-", label=crop.replace("_rf", ""))
        ax.set_xticks(range(len(SCHEMES))); ax.set_xticklabels(SCHEMES, rotation=20, fontsize=9)
        ax.set_title(met, fontsize=11); ax.grid(alpha=0.3)
    axes[0].legend(fontsize=8)
    fig.suptitle("Held-out scores of the SVM by validation scheme (random folds of 3.g = 8.2)", fontsize=12)
    fig.tight_layout(); fig.savefig(OUT / "block_cv_scores.png", dpi=100, bbox_inches="tight"); plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(18, 5))
    g = np.where(LAND, CONT, np.nan).astype(float)
    im = axes[0].pcolormesh(GRID["lon"], GRID["lat"], g, cmap="tab10", vmin=-0.5, vmax=9.5, shading="nearest")
    for k, c in enumerate(CONT_NAMES):
        axes[0].plot([], [], "s", color=plt.cm.tab10(k), label=c)
    axes[0].legend(fontsize=8, loc="lower left"); axes[0].set_ylim(-60, 85); axes[0].set_title("Continents (groups of AR6 regions)")
    sch = f"blocks {10 if 10 in BLOCK_DEGS else BLOCK_DEGS[0]} deg"
    H = FOLDMAP[(CROPS[0], sch)]
    gi = np.searchsorted(GRID["lat"], H.lat.values - 1e-6); gj = np.searchsorted(GRID["lon"], H.lon.values - 1e-6)
    gf = np.full(LAND.shape, np.nan); gf[gi, gj] = H.fold.values
    axes[1].pcolormesh(GRID["lon"], GRID["lat"], np.where(LAND, 0, np.nan), cmap="Greys", vmin=0, vmax=4, shading="nearest")
    axes[1].pcolormesh(GRID["lon"], GRID["lat"], gf, cmap="Set1", vmin=-0.5, vmax=8.5, shading="nearest")
    axes[1].set_ylim(-60, 85); axes[1].set_title(f"{sch}: fold of each training cell ({CROPS[0]})")
    fig.tight_layout(); fig.savefig(OUT / "block_cv_folds.png", dpi=90, bbox_inches="tight"); plt.show()

## 2. The two halves of ERA5: BIO variables (code of 12.a = 1.1.d) and final suitability (rule of 4.g)

Harvest years from `ERA5_PERIOD[0]` + 1 (as 1.1.d), split into H1 (the first half, rounded down) and H2. Check: full
reproduces the ERA5 BIO of 1.1.d.

In [ ]:
def compute_Ra(lat_rad, doy):
    Gsc = 0.0820
    dr = 1 + 0.033 * np.cos(2 * np.pi / 365 * doy)
    delta = 0.409 * np.sin(2 * np.pi / 365 * doy - 1.39)
    ws = np.arccos(np.clip(-np.tan(lat_rad) * np.tan(delta), -1, 1))
    return (24 * 60 / np.pi) * Gsc * dr * (ws * np.sin(lat_rad) * np.sin(delta) + np.cos(lat_rad) * np.cos(delta) * np.sin(ws))


def eto_day(tx, tn, lat_rad, doy):
    # compute_ETo_day of 1.1.d / 1.3.a (Hargreaves, Ra in mm/day equivalent)
    tm = (tx + tn) / 2.0
    return 0.0023 * compute_Ra(lat_rad, doy) / (2.501 - 0.002361 * tm) * (tm + 17.8) * np.sqrt(np.maximum(tx - tn, 0.0))


def ylen(y):
    return 366 if (y % 4 == 0 and (y % 100 != 0 or y % 400 == 0)) else 365


def bio_of(TMAX, TMIN, PR, ETO, dates, sow, ant, mat, harvest_years):
    # compute_bio_variables of 1.3.a on (days, cells) arrays, without radiation (no_rad): the BIO variables averaged
    # over the given harvest years (seasons crossing New Year start in hy - 1)
    n = sow.size
    yr_all, doy_all, mon_all = dates.year.values, dates.dayofyear.values, dates.month.values
    keys = (1, 2, 4, 5, 6, 7, 9, 10, 11, 12)
    s_, c_ = {k: np.zeros(n) for k in keys}, {k: np.zeros(n) for k in keys}
    valid = ~np.isnan(sow) & ~np.isnan(mat)
    wrap = valid & (mat > 365)
    present = set(yr_all)

    def acc(k, v):
        f = np.isfinite(v); s_[k] += np.where(f, v, 0); c_[k] += f

    for hy in harvest_years:
        if not (yr_all == hy).any():
            continue
        idx = np.where((yr_all == hy) | (yr_all == hy - 1))[0]
        yr3, doy3, mon3 = yr_all[idx][:, None], doy_all[idx][:, None], mon_all[idx][:, None]
        t3 = doy3 + np.where(yr3 == hy, ylen(hy - 1), 0)
        shift = np.where(wrap, 0, ylen(hy - 1))[None]
        s3, a3, m3 = np.nan_to_num(sow)[None] + shift, np.nan_to_num(ant)[None] + shift, np.nan_to_num(mat)[None] + shift
        vp = valid[None] & (((hy - 1) in present) | ~wrap[None])
        p1 = (t3 >= s3) & (t3 < a3)
        p2 = (t3 >= a3) & (t3 <= m3)
        gs = (p1 | p2) & vp; p1 &= vp; p2 &= vp
        tm, tn, pr, et = TMAX[idx], TMIN[idx], PR[idx], ETO[idx]
        pm = lambda a, m: np.nanmean(np.where(m, a, np.nan), axis=0)
        acc(1, pm(tm, p1)); acc(2, pm(tn, p1)); acc(4, pm(et, p1)); acc(5, pm(pr, p1))
        acc(6, pm(tm, p2)); acc(7, pm(tn, p2)); acc(9, pm(et, p2)); acc(10, pm(pr, p2))
        tmx, tmn = np.full(n, np.nan), np.full(n, np.nan)
        for m in range(1, 13):
            mm = gs & (mon3 == m)
            if not mm.any():
                continue
            tmx = np.fmax(tmx, np.nanmean(np.where(mm, tm, np.nan), axis=0))
            tmn = np.fmin(tmn, np.nanmean(np.where(mm, tn, np.nan), axis=0))
        acc(11, tmx); acc(12, tmn)
    out = {}
    for k in keys:
        with np.errstate(invalid="ignore", divide="ignore"):
            v = np.where(c_[k] > 0, s_[k] / c_[k], np.nan)
        v[~valid] = np.nan
        out[f"BIO{k:02d}"] = v
    return out


def grid_of(x):
    g = np.full(LAND.shape, np.nan, np.float32)
    g[I, J] = x
    return g


PARTS = ["full", "H1", "H2"]
if "halves" in SECTIONS:
    CAL = {}
    for crop in CROPS:
        with xr.open_dataset(era5_file(crop)) as e:
            CAL[crop] = {v: e[v].values for v in ("sowing_doy", "phase2_start_doy", "maturity_doy")}
    if OVERWRITE or not all(half_file(p, c).exists() for p in PARTS for c in CROPS):
        t0 = time.time()
        das = {v: cu.read_climate(CFG, CLIMATE, v, GRID, ERA5_PERIOD) for v in ("tasmax", "tasmin", "pr")}
        dates = pd.DatetimeIndex(das["tasmax"]["time"].values).normalize()
        TMAX, TMIN, PR = (cu.land_series(das[v], (I, J)) for v in ("tasmax", "tasmin", "pr"))
        doys = dates.dayofyear.values
        ETO = np.empty_like(TMAX)
        for t in range(dates.size):
            ETO[t] = eto_day(TMAX[t], TMIN[t], LAT_RAD, doys[t])
        hy = np.arange(int(ERA5_PERIOD[0]) + 1, int(ERA5_PERIOD[1]) + 1)       # harvest years of 1.1.d
        k = hy.size // 2
        P = {"full": hy, "H1": hy[:k], "H2": hy[k:]}
        HARVEST = {p: f"{P[p][0]}-{P[p][-1]} ({P[p].size})" if P[p].size else "none" for p in PARTS}
        print(f"{CLIMATE}: {dates[0].date()} -> {dates[-1].date()}, read in {(time.time() - t0) / 60:.1f} min | {HARVEST}", flush=True)
        for crop in CROPS:
            sow, ant, mat = (CAL[crop][v][I, J].astype(float) for v in ("sowing_doy", "phase2_start_doy", "maturity_doy"))
            for p in PARTS:
                b = bio_of(TMAX, TMIN, PR, ETO, dates, sow, ant, mat, P[p])
                ds = xr.Dataset({c: (("lat", "lon"), grid_of(b[c])) for c in COLS},
                                coords={"lat": GRID["lat"], "lon": GRID["lon"]},
                                attrs={"part": p, **{f"harvest_years_{q}": HARVEST[q] for q in PARTS},
                                       "description": f"BIO of {CLIMATE} {p}, code of 1.1.d / 12.a, {SET}"})
                for v in CAL[crop]:
                    ds[v] = (("lat", "lon"), CAL[crop][v])
                ds.to_netcdf(half_file(p, crop))
            print(f"  {crop} done", flush=True)
        del TMAX, TMIN, PR, ETO
    with xr.open_dataset(half_file("H1", CROPS[0])) as d:
        HARVEST = {p: d.attrs.get(f"harvest_years_{p}") for p in PARTS}
    print("harvest years:", HARVEST)
    rows = []                                   # check: full against 1.1.d (Antarctica apart: gap-filled calendar, no crop)
    for crop in CROPS:
        with xr.open_dataset(era5_file(crop)) as a, xr.open_dataset(half_file("full", crop)) as b:
            dd = {c: np.abs(a[c].values - b[c].values) for c in COLS}
            m = LAND & (LAT2 > -60)
            rows.append({"crop": crop, **{c: float(np.nanmax(dd[c][m])) for c in COLS},
                         "land cells differing > 1e-3": int(sum((np.nan_to_num(dd[c]) > 1e-3)[LAND] for c in COLS).astype(bool).sum()),
                         "of which south of 60 S": int(sum((np.nan_to_num(dd[c]) > 1e-3)[LAND & (LAT2 <= -60)] for c in COLS).astype(bool).sum())})
    print("|full - 1.1.d|: max north of 60 S (should be ~0) and cells differing:")
    display(pd.DataFrame(rows).set_index("crop"))

In [ ]:
def suitability(crop, files):
    # {key of files: {"suit": {level: bool (lat, lon)}, "novel": bool}}, as 4.g + 6.b (function of 12.a)
    bundle = joblib.load(MODELS / f"svm_model_{TAG}_{crop}.joblib")
    cols, n1, n0 = bundle["bio_cols"], bundle["n_class1"], bundle["n_class0"]
    assert set(cols) <= set(COLS), (cols, COLS)
    tr = pd.read_csv(ERA5_DIR / f"training_dataset_{TAG}_{crop}.csv")
    ii = np.searchsorted(GRID["lat"], tr.lat.values - 1e-6); jj = np.searchsorted(GRID["lon"], tr.lon.values - 1e-6)
    with xr.open_dataset(era5_file(crop)) as ds:
        REF = cu.bio_array(ds, cols, CFG)[ii, jj]
    REF = REF[np.isfinite(REF).all(axis=1)]
    kfun = cu.kernel_support(bundle["model"])
    Fd = cu.crossfit_models(MODELS / f"folds_{TAG}_{crop}.joblib", tr[cols].values.astype(float),
                            tr["label"].values.astype(int), bundle, n_jobs=N_JOBS)
    ssr = {lv: np.where(LAND, np.nan_to_num(SOIL[lv][crop].values), 0.0) for lv in LEVELS}
    which = cu.ECOCROP_TMIN[crop][1]
    out = {}
    for key, path in files.items():
        with xr.open_dataset(path) as ds:
            A = cu.bio_array(ds, cols, CFG)
            tg = np.asarray(cu.growing_temperature(ds, which), float)
        X = A.reshape(-1, len(cols))
        ok = np.isfinite(X).all(axis=1) & LAND.ravel()
        p, nov = np.full(X.shape[0], np.nan), np.zeros(X.shape[0], bool)
        if ok.any():                                              # (a part without any season: all NaN)
            Xs = bundle["scaler"].transform(X[ok])
            p[ok] = bundle["model"].predict_proba(Xs)[:, 1]
            nov[ok] = (cu.mess(X[ok], REF)[0] < 0) | (cu.nt2(X[ok], REF) > 1) | (kfun(Xs) < KERNEL_EPS)
        p, nov = p.reshape(LAND.shape), nov.reshape(LAND.shape)
        g = cu.crossfit_proba(Fd, A[ii, jj])                      # training cells: the fold model that did not see them
        fin = np.isfinite(g)
        p[ii[fin], jj[fin]] = g[fin]
        p = cu.balanced_proba(p, n1, n0)
        nov &= np.isfinite(p)
        proba = np.where(nov, 0.0, p)
        clim = cu.climate_suitable(proba, tg, crop, DT) & np.isfinite(proba)
        out[key] = {"suit": {lv: LAND & cu.suitable(clim, ssr[lv]) for lv in LEVELS}, "novel": nov}
    return out


if "halves" in SECTIONS:
    S, TSSH = {}, []
    for crop in CROPS:
        t0 = time.time()
        S[crop] = suitability(crop, {"ERA5": era5_file(crop), **{p: half_file(p, crop) for p in PARTS}})
        # TSS of the training cells with the climate of each part (cross-fitting, equal priors, no novelty: training cells)
        bundle = joblib.load(MODELS / f"svm_model_{TAG}_{crop}.joblib")
        cols = bundle["bio_cols"]
        tr = pd.read_csv(ERA5_DIR / f"training_dataset_{TAG}_{crop}.csv")
        y = tr["label"].values.astype(int)
        ii = np.searchsorted(GRID["lat"], tr.lat.values - 1e-6); jj = np.searchsorted(GRID["lon"], tr.lon.values - 1e-6)
        Fd = cu.crossfit_models(MODELS / f"folds_{TAG}_{crop}.joblib", tr[cols].values.astype(float), y, bundle, n_jobs=N_JOBS)
        for p in ["ERA5", *PARTS]:
            with xr.open_dataset(era5_file(crop) if p == "ERA5" else half_file(p, crop)) as ds:
                A = cu.bio_array(ds, cols, CFG)[ii, jj]
            g = cu.balanced_proba(cu.crossfit_proba(Fd, A), bundle["n_class1"], bundle["n_class0"])
            ok = np.isfinite(g)
            if not ((y[ok] == 1).any() and (y[ok] == 0).any()):
                continue                                          # a part without any season
            s = g >= cu.CLIMATE_P
            TSSH.append({"crop": crop, "part": p, "cells": int(ok.sum()),
                         "TSS (training cells)": s[ok & (y == 1)].mean() + (~s[ok & (y == 0)]).mean() - 1,
                         "AUC (training cells)": roc_auc_score(y[ok], g[ok])})
        print(f"{crop}: {(time.time() - t0) / 60:.1f} min", flush=True)

    rows = []                                   # check: ERA5 (1.1.d) and full give the same suitability
    for crop in CROPS:
        for lv in LEVELS:
            rows.append({"crop": crop, "level": lv,
                         "cells differing, full vs ERA5": int((S[crop]["full"]["suit"][lv] != S[crop]["ERA5"]["suit"][lv])[LAND].sum())})
    print("full against ERA5 of 1.1.d (should be 0, up to float rounding):")
    display(pd.DataFrame(rows).pivot(index="crop", columns="level", values="cells differing, full vs ERA5"))

## 3. Halves of ERA5: today's crop area suitable, suitable land, cells that change class

`today's crop area` = share of the MIRCA2000 area of the crop in suitable cells (%); `<region> land` = suitable land
(Mha). H2 − H1 in points / %; SE of the 33-season baseline ≈ |H2 − H1| / 2. Flips: share of today's crop area and land
that change class between H1 and H2 (cells not novel in either). Comparison: effect of the AMOC weakening of 12.a
(full, 2050) where its output exists.

In [ ]:
if "halves" in SECTIONS:
    mha = lambda m: float(np.nansum(LAND_MHA * m))
    rows, flips = [], []
    for crop in CROPS:
        crop_ha = np.nan_to_num(cu.crop_mask(CFG, crop)["area_ha"].values)
        share = lambda m: 100 * crop_ha[m].sum() / crop_ha.sum()
        R = S[crop]
        for lv in LEVELS:
            for p in ["ERA5", *PARTS]:
                r = {"crop": crop, "level": lv, "part": p, "today's crop area (%)": share(R[p]["suit"][lv]),
                     "novel land (Mha)": mha(R[p]["novel"])}
                for reg, rm in REGIONS.items():
                    r[f"{reg} land (Mha)"] = mha(R[p]["suit"][lv] & rm)
                rows.append(r)
            A, B = R["H2"], R["H1"]
            ok = ~(A["novel"] | B["novel"])
            ch = (A["suit"][lv] != B["suit"][lv]) & ok
            flips.append({"crop": crop, "level": lv,
                          "today's crop area flipped (%)": share(ch),
                          "of which suitable in H2 only (%)": share(ch & A["suit"][lv]),
                          "land flipped (Mha)": mha(ch & LAND),
                          "land suitable in H2 only (Mha)": mha(ch & LAND & A["suit"][lv])})
    HALF = pd.DataFrame(rows).set_index(["crop", "level", "part"])
    FL = pd.DataFrame(flips).set_index(["crop", "level"])
    TS = pd.DataFrame(TSSH).set_index(["crop", "part"])
    D21 = (HALF.xs("H2", level="part") - HALF.xs("H1", level="part"))
    for c in [c for c in D21.columns if c.endswith("land (Mha)")]:
        D21[c.replace("(Mha)", "(%)")] = 100 * D21[c] / HALF.xs("H1", level="part")[c].where(HALF.xs("H1", level="part")[c] > 0.5)
    D21["SE of the baseline, today's crop area (points)"] = D21["today's crop area (%)"].abs() / 2
    HALF.to_csv(OUT / "halves_suitability.csv"); FL.to_csv(OUT / "halves_flips.csv"); TS.to_csv(OUT / "halves_tss.csv")
    D21.to_csv(OUT / "halves_difference.csv")
    with pd.option_context("display.max_rows", 400, "display.width", 250):
        print("Suitability of each part")
        display(HALF[["today's crop area (%)", "global land (Mha)", "north of 50 N land (Mha)", "Europe land (Mha)", "novel land (Mha)"]].round(1))
        print("H2 - H1 (points of today's crop area; % of the suitable land of H1)")
        display(D21[["today's crop area (%)", "SE of the baseline, today's crop area (points)", "global land (%)",
                     "north of 50 N land (%)", "Europe land (%)", "tropics land (%)"]].round(2))
        print("Cells changing class between H1 and H2")
        display(FL.round(2))
        print("Skill on the training cells with the climate of each part")
        display(TS.round(3))
    # comparison with the effect of the AMOC weakening (12.a), 2050, today's crop area
    f12 = VAR12A / "effect_vs_noise.csv"
    if f12.exists():
        E = pd.read_csv(f12)
        E = E[(E.metric == "today's crop area (points)") & E.pair.str.startswith("gcm_2050")]
        E = E.set_index(["crop", "level"])[["full", "SE (noise)"]].rename(columns={"full": "AMOC weakening 2050 (points)",
                                                                                  "SE (noise)": "SE of the runs, 12.a (points)"})
        CMP = E.join(D21[["today's crop area (%)", "SE of the baseline, today's crop area (points)"]].rename(
            columns={"today's crop area (%)": "ERA5 H2 - H1 (points)"}))
        CMP.to_csv(OUT / "halves_vs_amoc_effect.csv")
        print("Today's crop area: ERA5 H2 - H1 against the effect of the AMOC weakening in 2050 (12.a)")
        display(CMP.round(2))
    else:
        print("no output of 12.a: comparison skipped")

## 4. Maps: cells that change class between the two halves of ERA5 (high input)

In [ ]:
if "halves" in SECTIONS:
    LV = "high" if "high" in LEVELS else LEVELS[0]
    from matplotlib.colors import ListedColormap
    cmap = ListedColormap(["#8c8c8c", "#4d9221", "#c51b7d", "#2166ac"])   # suitable in both, H1 only, H2 only, novel
    for crop in CROPS:
        R = S[crop]
        a, b = R["H1"]["suit"][LV], R["H2"]["suit"][LV]
        nov = R["H1"]["novel"] | R["H2"]["novel"]
        g = np.full(LAND.shape, np.nan)
        g[a & b] = 0; g[a & ~b & ~nov] = 1; g[~a & b & ~nov] = 2; g[nov & LAND] = 3
        fig, ax = plt.subplots(figsize=(12, 5.2))
        ax.pcolormesh(GRID["lon"], GRID["lat"], np.where(LAND, 1, np.nan), cmap="Greys", vmin=0, vmax=8, shading="nearest")
        ax.pcolormesh(GRID["lon"], GRID["lat"], g, cmap=cmap, vmin=-0.5, vmax=3.5, shading="nearest")
        for k, lab in enumerate(["suitable in both halves", f"suitable in {HARVEST['H1'].split()[0]} only", f"suitable in {HARVEST['H2'].split()[0]} only", "novel in either half"]):
            ax.plot([], [], "s", color=cmap(k), label=lab)
        f = FL.loc[(crop, LV)]
        fa = f["today's crop area flipped (%)"]
        ax.legend(fontsize=8, loc="lower left"); ax.set_ylim(-60, 85)
        ax.set_title(f"{crop}, {LV} input: the two halves of ERA5 ({HARVEST['H1']} / {HARVEST['H2']}); today's crop area "
                     f"changing class {fa:.1f} %, land {f['land flipped (Mha)']:.0f} Mha", fontsize=10)
        fig.tight_layout(); fig.savefig(OUT / f"halves_map_{crop}.png", dpi=90, bbox_inches="tight")
        plt.show() if crop == CROPS[0] else plt.close(fig)
print("written:", sorted(p.name for p in OUT.iterdir()))